# Views and Materialised Views

## Overview
A **view** is a named, saved SELECT query. Querying a view runs the underlying SELECT at that moment -- no data is stored. It looks and feels like a table but is always computed fresh.

```sql
CREATE VIEW active_patients AS
SELECT patient_id, first_name, last_name, province
FROM   patients
WHERE  active = 1;

-- Usage: treated exactly like a table
SELECT * FROM active_patients WHERE province = 'ON';
```

**View use cases:**
- Hiding complexity (pre-joined tables, calculated columns)
- Security layer: expose only certain columns/rows to a role (see folder 23)
- Stable interface: the underlying table can change without breaking downstream queries that use the view
- Readability: give business-meaningful names to complex query logic

**Materialised views (PostgreSQL only):**
A materialised view stores the query result physically -- like a cached table snapshot. Reads are instant; data must be refreshed manually with `REFRESH MATERIALIZED VIEW`.

| | Regular view | Materialised view |
|---|---|---|
| Data stored | No (computed on each query) | Yes (snapshot on disk) |
| Always current | Yes | No (stale until refreshed) |
| Can be indexed | No | Yes |
| Best for | Simple lookups, security layers | Slow queries run frequently |
| Refresh needed | Never | After source data changes |

**SQLite note:** SQLite supports `CREATE VIEW` but does NOT support materialised views. Materialised view syntax and behaviour shown as PostgreSQL-only examples.

---

In [1]:
import sqlite3
import pandas as pd
conn = sqlite3.connect(":memory:")
conn.executescript("""CREATE TABLE patients (patient_id INTEGER PRIMARY KEY, first_name TEXT NOT NULL, last_name TEXT NOT NULL, dob TEXT NOT NULL, sex TEXT CHECK(sex IN ('M','F','O')), province TEXT, active INTEGER DEFAULT 1);CREATE TABLE departments (dept_id INTEGER PRIMARY KEY, dept_name TEXT NOT NULL UNIQUE, building TEXT, floor_no INTEGER);CREATE TABLE providers (provider_id INTEGER PRIMARY KEY, full_name TEXT NOT NULL, specialty TEXT, licence_no TEXT UNIQUE, province TEXT, dept_id INTEGER REFERENCES departments(dept_id), manager_id INTEGER REFERENCES providers(provider_id));CREATE TABLE diagnoses (diag_code TEXT PRIMARY KEY, description TEXT NOT NULL, category TEXT NOT NULL);CREATE TABLE encounters (enc_id INTEGER PRIMARY KEY, patient_id INTEGER NOT NULL REFERENCES patients(patient_id), provider_id INTEGER NOT NULL REFERENCES providers(provider_id), dept_id INTEGER REFERENCES departments(dept_id), enc_date TEXT NOT NULL, cost_cad REAL, admitted INTEGER DEFAULT 0 CHECK(admitted IN (0,1)));CREATE TABLE encounter_diagnoses (enc_id INTEGER NOT NULL REFERENCES encounters(enc_id), diag_code TEXT NOT NULL REFERENCES diagnoses(diag_code), diag_rank INTEGER DEFAULT 1, confirmed INTEGER DEFAULT 1, PRIMARY KEY (enc_id, diag_code));CREATE TABLE lab_results (result_id INTEGER PRIMARY KEY, patient_id INTEGER NOT NULL REFERENCES patients(patient_id), test_name TEXT NOT NULL, result_val REAL, units TEXT, collected TEXT NOT NULL, flagged INTEGER DEFAULT 0);INSERT INTO departments VALUES (1,'Emergency','Tower A',1),(2,'Cardiology','Tower B',2),(3,'Oncology','Tower C',3),(4,'Family Medicine','Clinic',1),(5,'Orthopaedics','Tower A',2),(6,'Radiology','Tower B',3),(7,'ICU','Tower C',NULL);INSERT INTO providers VALUES (10,'Dr. Sarah MacNeil','Cardiology','MC-1001','NB',2,NULL),(11,'Dr. James Wong','Oncology','MC-1002','BC',3,10),(12,'Dr. Fatima Osei','Family Medicine','MC-1003','ON',4,10),(13,'Dr. Ethan Larson','Orthopaedics','MC-1004','QC',5,10),(14,'Dr. Priya Sharma','Emergency','MC-1005','AB',1,10),(15,'Dr. Lucas Petit','Radiology','MC-1006','QC',6,11);INSERT INTO patients VALUES (1,'Aroha','Ngata','1985-03-12','F','NB',1),(2,'Liam','Chen','1972-11-04','M','NS',1),(3,'Fatima','Al-Rashid','1990-07-22','F','ON',1),(4,'James','MacLeod','1955-01-30','M','NB',0),(5,'Sofia','Petrov','2001-09-15','F','BC',1),(6,'Noah','Williams','1968-06-08','M','AB',1),(7,'Mei','Zhang','1995-04-17','F','ON',1),(8,'Ethan','Tremblay','1980-12-01','M','QC',0),(9,'Priya','Nair','1977-08-25','F','BC',1),(10,'Marcus','Okafor','1993-05-19','M','ON',1),(11,'Diana','Flores','2000-02-14','F','NB',1);INSERT INTO diagnoses VALUES ('I25.1','Atherosclerotic heart disease','Cardiovascular'),('J18.9','Pneumonia, unspecified','Respiratory'),('Z00.0','General medical examination','Preventive'),('M17.1','Primary osteoarthritis of knee','Musculoskeletal'),('J06.9','Acute upper respiratory infection','Respiratory'),('R07.9','Chest pain, unspecified','Symptoms'),('I10','Essential hypertension','Cardiovascular'),('R55','Syncope and collapse','Symptoms'),('I48.0','Paroxysmal atrial fibrillation','Cardiovascular'),('S52.5','Fracture of lower end of radius','Injury'),('E11.9','Type 2 diabetes mellitus','Endocrine'),('I50.9','Heart failure, unspecified','Cardiovascular');INSERT INTO encounters VALUES (1,1,10,2,'2023-01-15',3200.00,1),(2,2,14,1,'2023-02-20',1850.00,1),(3,3,12,4,'2023-03-05',120.00,0),(4,4,13,5,'2023-03-18',5500.00,1),(5,5,12,4,'2023-04-02',95.00,0),(6,6,14,1,'2023-05-11',780.00,0),(7,7,10,2,'2023-06-22',2100.00,1),(8,8,12,4,'2023-07-14',80.00,0),(9,1,14,1,'2023-08-30',410.00,0),(10,3,12,4,'2023-09-12',110.00,0),(11,9,10,2,'2023-10-01',1750.00,1),(12,10,14,1,'2023-11-03',2200.00,1),(13,2,10,2,'2023-11-20',2900.00,1),(14,6,12,4,'2023-12-01',115.00,0);INSERT INTO encounter_diagnoses VALUES (1,'I25.1',1,1),(1,'I10',2,1),(2,'J18.9',1,1),(3,'Z00.0',1,1),(4,'M17.1',1,1),(5,'J06.9',1,1),(6,'R07.9',1,1),(7,'I10',1,1),(7,'I48.0',2,1),(9,'R55',1,1),(10,'Z00.0',1,1),(11,'I48.0',1,1),(12,'S52.5',1,1),(13,'I25.1',1,1),(14,'Z00.0',1,1);INSERT INTO lab_results VALUES (1,1,'HbA1c',7.2,'%','2023-03-10',0),(2,2,'HbA1c',9.1,'%','2023-03-15',1),(3,3,'Creatinine',88.5,'umol/L','2023-04-01',0),(4,4,'Creatinine',145.0,'umol/L','2023-04-12',1),(5,5,'HbA1c',5.8,'%','2023-05-05',0),(6,6,'Sodium',138.0,'mmol/L','2023-05-20',0),(7,7,'Sodium',151.0,'mmol/L','2023-06-01',1),(8,1,'Creatinine',72.3,'umol/L','2023-06-18',0),(9,8,'HbA1c',6.4,'%','2023-07-02',0),(10,9,'Creatinine',210.0,'umol/L','2023-07-15',1),(11,2,'Creatinine',95.0,'umol/L','2023-08-01',0),(12,10,'HbA1c',7.8,'%','2023-08-20',1);""")
conn.commit()
print("Healthcare schema ready.")

Healthcare schema ready.


---
## Creating and querying views

In [2]:
# Create several useful views over the healthcare schema
conn.executescript("""
-- Active patients only
CREATE VIEW active_patients AS
SELECT patient_id, first_name, last_name, dob, sex, province
FROM   patients WHERE active = 1;

-- Encounters enriched with patient and provider names
CREATE VIEW encounter_detail AS
SELECT e.enc_id, e.enc_date,
       p.first_name || ' ' || p.last_name  AS patient_name,
       p.province                           AS patient_province,
       pr.full_name                         AS provider_name,
       pr.specialty,
       d.dept_name,
       e.cost_cad,
       e.admitted
FROM   encounters AS e
JOIN   patients     AS p  ON e.patient_id  = p.patient_id
JOIN   providers    AS pr ON e.provider_id = pr.provider_id
LEFT JOIN departments AS d ON e.dept_id    = d.dept_id;

-- Patient encounter summary (aggregated view)
CREATE VIEW patient_summary AS
SELECT p.patient_id,
       p.first_name || ' ' || p.last_name  AS patient_name,
       p.province,
       COUNT(e.enc_id)                      AS total_encounters,
       ROUND(SUM(e.cost_cad), 2)            AS total_cost,
       SUM(e.admitted)                      AS admissions,
       MAX(e.enc_date)                      AS last_encounter
FROM   patients AS p
LEFT JOIN encounters AS e ON p.patient_id = e.patient_id
GROUP BY p.patient_id;
""")
conn.commit()

print("=== Views created ===")
views = conn.execute("SELECT name FROM sqlite_master WHERE type='view'").fetchall()
for (v,) in views:
    print(f"  {v}")

print()
print("=== Querying active_patients view ===")
print(pd.read_sql("SELECT * FROM active_patients ORDER BY province, last_name", conn).to_string(index=False))


=== Views created ===
  active_patients
  encounter_detail
  patient_summary

=== Querying active_patients view ===
 patient_id first_name last_name        dob sex province
          6       Noah  Williams 1968-06-08   M       AB
          9      Priya      Nair 1977-08-25   F       BC
          5      Sofia    Petrov 2001-09-15   F       BC
         11      Diana    Flores 2000-02-14   F       NB
          1      Aroha     Ngata 1985-03-12   F       NB
          2       Liam      Chen 1972-11-04   M       NS
          3     Fatima Al-Rashid 1990-07-22   F       ON
         10     Marcus    Okafor 1993-05-19   M       ON
          7        Mei     Zhang 1995-04-17   F       ON


---
## View as abstraction layer

In [3]:
print("=== encounter_detail view hides join complexity ===")
q = """
SELECT enc_id, enc_date, patient_name, provider_name, specialty, dept_name, cost_cad, admitted
FROM   encounter_detail
WHERE  admitted = 1
ORDER BY enc_date
"""
print(pd.read_sql(q, conn).to_string(index=False))

print()
print("=== patient_summary view: aggregated without writing GROUP BY every time ===")
q2 = """
SELECT patient_name, province, total_encounters, total_cost, admissions, last_encounter
FROM   patient_summary
WHERE  total_encounters > 1
ORDER BY total_cost DESC
"""
print(pd.read_sql(q2, conn).to_string(index=False))


=== encounter_detail view hides join complexity ===
 enc_id   enc_date  patient_name     provider_name    specialty    dept_name  cost_cad  admitted
      1 2023-01-15   Aroha Ngata Dr. Sarah MacNeil   Cardiology   Cardiology    3200.0         1
      2 2023-02-20     Liam Chen  Dr. Priya Sharma    Emergency    Emergency    1850.0         1
      4 2023-03-18 James MacLeod  Dr. Ethan Larson Orthopaedics Orthopaedics    5500.0         1
      7 2023-06-22     Mei Zhang Dr. Sarah MacNeil   Cardiology   Cardiology    2100.0         1
     11 2023-10-01    Priya Nair Dr. Sarah MacNeil   Cardiology   Cardiology    1750.0         1
     12 2023-11-03 Marcus Okafor  Dr. Priya Sharma    Emergency    Emergency    2200.0         1
     13 2023-11-20     Liam Chen Dr. Sarah MacNeil   Cardiology   Cardiology    2900.0         1

=== patient_summary view: aggregated without writing GROUP BY every time ===
    patient_name province  total_encounters  total_cost  admissions last_encounter
       Liam

---
## View limitations and updatable views

In [4]:
# Views cannot be indexed in SQLite; they recompute on every query
# Complex views (with aggregation) cannot be updated through the view

print("=== Attempting UPDATE through a simple view ===")
try:
    conn.execute("UPDATE active_patients SET province = 'BC' WHERE patient_id = 1")
    conn.commit()
    result = conn.execute("SELECT province FROM patients WHERE patient_id=1").fetchone()[0]
    print(f"Update succeeded -- patient_id=1 province is now: {result}")
except Exception as e:
    print(f"Update failed: {e}")

print()
print("=== Simple views (no aggregation, no JOIN) are updatable in PostgreSQL ===")
print("""PostgreSQL updatable view rules:
  - No DISTINCT, GROUP BY, HAVING, LIMIT in the view
  - No set operations (UNION, INTERSECT, EXCEPT)
  - No aggregate or window functions
  - Based on a single table (or WITH CHECK OPTION for multi-table)
  - All NOT NULL columns with no DEFAULT must be included

For non-updatable views, use INSTEAD OF triggers or rules in PostgreSQL.
""")

print("=== Materialised views (PostgreSQL only) ===")
print("""
  -- Create a materialised view (stores results on disk):
  CREATE MATERIALIZED VIEW patient_cost_summary AS
  SELECT patient_id,
         SUM(cost_cad)    AS total_cost,
         COUNT(*)         AS enc_count,
         MAX(enc_date)    AS last_enc
  FROM   encounters
  GROUP BY patient_id
  WITH DATA;  -- WITH DATA computes immediately; WITH NO DATA defers

  -- Add an index on the materialised view:
  CREATE INDEX ON patient_cost_summary (patient_id);
  CREATE INDEX ON patient_cost_summary (total_cost DESC);

  -- Refresh when source data changes (blocks reads in basic mode):
  REFRESH MATERIALIZED VIEW patient_cost_summary;

  -- Refresh without blocking reads (requires unique index):
  REFRESH MATERIALIZED VIEW CONCURRENTLY patient_cost_summary;

  -- Drop when no longer needed:
  DROP MATERIALIZED VIEW patient_cost_summary;
""")
conn.close()


=== Attempting UPDATE through a simple view ===
Update failed: cannot modify active_patients because it is a view

=== Simple views (no aggregation, no JOIN) are updatable in PostgreSQL ===
PostgreSQL updatable view rules:
  - No DISTINCT, GROUP BY, HAVING, LIMIT in the view
  - No set operations (UNION, INTERSECT, EXCEPT)
  - No aggregate or window functions
  - Based on a single table (or WITH CHECK OPTION for multi-table)
  - All NOT NULL columns with no DEFAULT must be included

For non-updatable views, use INSTEAD OF triggers or rules in PostgreSQL.

=== Materialised views (PostgreSQL only) ===

  -- Create a materialised view (stores results on disk):
  CREATE MATERIALIZED VIEW patient_cost_summary AS
  SELECT patient_id,
         SUM(cost_cad)    AS total_cost,
         COUNT(*)         AS enc_count,
         MAX(enc_date)    AS last_enc
  FROM   encounters
  GROUP BY patient_id
  WITH DATA;  -- WITH DATA computes immediately; WITH NO DATA defers

  -- Add an index on the materi

---
## Common Pitfalls

**1. Assuming views are always fast**
A regular view re-executes the full underlying query every time it is queried. A complex view joining 6 tables with aggregation runs all 6 joins and the GROUP BY on every query -- no caching. If the view is slow, the query is slow. Use a materialised view or a summary table for expensive calculations queried frequently.

**2. Building views on top of views**
`CREATE VIEW v3 AS SELECT * FROM v2 JOIN v1` -- the database expands all nested views into one giant query plan. Three layers of views can produce a plan the optimiser cannot optimise well. Keep view depth shallow; use CTEs instead for complex multi-step logic.

**3. Materialised views go stale silently**
After inserting 1000 new encounters, the `patient_cost_summary` materialised view still shows old totals until you run `REFRESH MATERIALIZED VIEW`. There is no automatic refresh in standard PostgreSQL. Build refresh into your ETL pipeline or use triggers.

**4. Refreshing a materialised view blocks all reads**
`REFRESH MATERIALIZED VIEW` takes an exclusive lock -- concurrent queries against the view queue up until the refresh completes. For production, use `REFRESH MATERIALIZED VIEW CONCURRENTLY` (requires a unique index on the materialised view).

**5. Views do not improve query performance for simple queries**
`SELECT * FROM encounter_detail WHERE enc_id = 5` is not faster than writing the join directly. The view is syntactic sugar, not a cached result. For performance, the underlying tables need proper indexes.

**6. Dropping a base table that a view depends on**
`DROP TABLE encounters` fails with "view encounter_detail depends on table encounters" in PostgreSQL. Use `DROP TABLE encounters CASCADE` to drop the table and all dependent views -- but this silently removes the views. Check `pg_depend` or `\dv` to audit view dependencies before dropping tables.


---
*sql_methods_library - Samantha McGarrigle*